# Task 2. `eth_getLogs` Transfer 샘플 검증

## 검증 의도

현재 Python RPC client로 `eth_getLogs` 호출이 가능한지 확인하고, ERC-20 Transfer event의 `topic0`, `topic1`, `topic2`, `data` 해석 흐름을 DataFrame으로 확인합니다.
이 노트북은 수집 파이프라인 전체 실행이 아니라 provider 응답과 decode 규칙을 눈으로 검증하는 smoke 자료입니다.

## 검증 경계

- sample contract는 WETH입니다. 과제의 Tether Treasury 집계 범위를 넓힌다는 의미가 아닙니다.
- 실제 수집 범위는 `PipelineSettings.collection_scope`와 Airflow DAG 기준입니다.
- 외부 RPC credential이 없으면 `BLOCKED`로 남깁니다.

In [7]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    """노트북 실행 위치와 무관하게 현재 repository root를 찾는다."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "cryptoquant_pipeline").exists():
            return candidate
    raise RuntimeError("repository root를 찾지 못함. pyproject.toml과 src/cryptoquant_pipeline 확인 필요함.")


PROJECT_ROOT = find_repo_root()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

ENV_PATH = PROJECT_ROOT / ".env"
if ENV_PATH.exists():
    load_dotenv(ENV_PATH)

SAFE_ENV_STATUS = {
    "project_root": str(PROJECT_ROOT),
    "env_file_exists": ENV_PATH.exists(),
    "eth_rpc_url_configured": bool(os.getenv("ETH_RPC_URL") or os.getenv("CHAINSTACK_RPC_URL")),
    "auth_mode": os.getenv("ETH_RPC_AUTH_MODE") or os.getenv("CHAINSTACK_AUTH_MODE") or "none",
}

display(Markdown("### Repository 및 환경변수 확인"))
display(SAFE_ENV_STATUS)

### Repository 및 환경변수 확인

{'project_root': '/workspace',
 'env_file_exists': True,
 'eth_rpc_url_configured': True,
 'auth_mode': 'none'}

In [8]:
from cryptoquant_pipeline.config import PipelineSettings
from cryptoquant_pipeline.exceptions import ConfigError

settings = None
settings_status = "READY"
settings_error = None
try:
    settings = PipelineSettings.from_env()
except ConfigError as exc:
    settings_status = "BLOCKED"
    settings_error = f"{exc.__class__.__name__}: {exc}"

safe_settings = {
    "status": settings_status,
    "error": settings_error,
    "chain_id": None if settings is None else settings.chain_id,
    "auth_mode": None if settings is None else settings.provider.auth_mode,
    "provider_configured": settings is not None,
    "max_blocks_per_log_request": None if settings is None else settings.max_blocks_per_log_request,
    "requests_per_second": None if settings is None else settings.rpc_requests_per_second,
    "delta_logs_path": None if settings is None else str(settings.delta_logs_path),
    "duckdb_path": None if settings is None else str(settings.duckdb_path),
    "collection_scope": None if settings is None else settings.collection_scope.scope_id,
    "collection_scope_fingerprint": None if settings is None else settings.collection_scope.fingerprint,
}

display(Markdown("### Python 설정 객체 검증"))
display(safe_settings)

### Python 설정 객체 검증

{'status': 'READY',
 'error': None,
 'chain_id': 1,
 'auth_mode': 'none',
 'provider_configured': True,
 'max_blocks_per_log_request': 10,
 'requests_per_second': 4.0,
 'delta_logs_path': '/opt/airflow/data/delta/ethereum_logs_v2',
 'duckdb_path': '/opt/airflow/data/analytics/ethereum_analytics_v2.duckdb',
 'collection_scope': 'transfer_topic_all_addresses',
 'collection_scope_fingerprint': 'af0864e874141de6657364a36407791a0769dee5c55679b2089c4efb33e8b885'}

In [9]:
from decimal import Decimal

import pandas as pd

from cryptoquant_pipeline.config import TRANSFER_TOPIC0
from cryptoquant_pipeline.log_normalizer import amount_with_decimals, decode_uint256_decimal, topic_to_address
from cryptoquant_pipeline.rpc_client import EthereumJsonRpcClient

WETH_MAINNET_ADDRESS = "0xC02aaA39b223FE8D0A0E5C4F27eAD9083C756Cc2"
WETH_DECIMALS = 18
BLOCK_WINDOW = 10

if settings is None:
    display(Markdown("### eth_getLogs skipped\n`ETH_RPC_URL`이 없어 실제 provider 호출을 실행하지 않았습니다."))
    df_logs = pd.DataFrame()
else:
    with EthereumJsonRpcClient(
        settings.provider,
        timeout_seconds=settings.rpc_timeout_seconds,
        max_retries=settings.rpc_max_retries,
        requests_per_second=settings.rpc_requests_per_second,
    ) as client:
        finalized_block = client.eth_get_finalized_block()
        finalized_block_number = int(finalized_block["number"], 16)
        from_block = max(0, finalized_block_number - BLOCK_WINDOW + 1)
        to_block = finalized_block_number
        logs = client.eth_get_logs(
            from_block=from_block,
            to_block=to_block,
            address=WETH_MAINNET_ADDRESS,
            topics=[TRANSFER_TOPIC0],
        )

    records = []
    for log in logs:
        raw_amount = decode_uint256_decimal(str(log["data"]))
        records.append(
            {
                "block_number": int(str(log["blockNumber"]), 16),
                "transaction_hash": str(log["transactionHash"]),
                "log_index": int(str(log["logIndex"]), 16),
                "contract_address": str(log["address"]).lower(),
                "topic0": str(log["topics"][0]).lower(),
                "from_address": topic_to_address(str(log["topics"][1])),
                "to_address": topic_to_address(str(log["topics"][2])),
                "value_raw_decimal_text": str(raw_amount),
                "value_weth_decimal_text": str(amount_with_decimals(raw_amount, WETH_DECIMALS)),
            }
        )

    df_logs = pd.DataFrame(records)
    summary = {
        "from_block": from_block,
        "to_block": to_block,
        "contract": WETH_MAINNET_ADDRESS.lower(),
        "topic0": TRANSFER_TOPIC0,
        "log_count": len(df_logs),
        "unique_transaction_count": 0 if df_logs.empty else int(df_logs["transaction_hash"].nunique()),
    }
    display(Markdown("### eth_getLogs 호출 결과"))
    display(summary)
    display(df_logs.head(20))

    if not df_logs.empty and (df_logs["topic0"] != TRANSFER_TOPIC0).any():
        raise RuntimeError("Transfer topic0가 아닌 log가 포함됨.")

### eth_getLogs 호출 결과

{'from_block': 25373918,
 'to_block': 25373927,
 'contract': '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2',
 'topic0': '0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4a11628f55a4df523b3ef',
 'log_count': 306,
 'unique_transaction_count': 175}

,block_number,transaction_hash,log_index,contract_address,topic0,from_address,to_address,value_raw_decimal_text,value_weth_decimal_text
0,25373918,0xaaf6917e42ec0d2e8f9631c6e464d1cfa555ec98a905...,0,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x51c72848c68a965f66fa7a88855f9f7784502a7f,0x5386512190c0c83c608dc0e8ce1fbe7644de07e2,5242966132591810560,5.24296613259181056
1,25373918,0xaaf6917e42ec0d2e8f9631c6e464d1cfa555ec98a905...,1,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x5386512190c0c83c608dc0e8ce1fbe7644de07e2,0xa250cc729bb3323e7933022a67b52200fe354767,5242966132591810560,5.24296613259181056
2,25373918,0xfd4ec41ecda1e94ce18eb17d1c3aa65563a65cf2bb01...,11,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x51c72848c68a965f66fa7a88855f9f7784502a7f,0x1d42064fc4beb5f8aaf85f4617ae8b3b5b8bd801,4356773324998206464,4.356773324998206464
3,25373918,0xda84198e8920bd1bea7171b2b9be3cd71f5579622592...,14,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x51c72848c68a965f66fa7a88855f9f7784502a7f,0xba47cbfdd61029833841fcaa2ec2591ddfa87e51,968481193472193664,0.968481193472193664
4,25373918,0x2914e5292658047fca510fcb99c0d1f4c440d1fdeaee...,20,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0xba47cbfdd61029833841fcaa2ec2591ddfa87e51,0x51c72848c68a965f66fa7a88855f9f7784502a7f,3382440971150524892,3.382440971150524892
5,25373918,0x8ef1d5b0a63ad1ef6df97b3a03e86484b65b73c4e812...,24,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x11b815efb8f581194ae79006d24e0d814b7697f6,0x51c72848c68a965f66fa7a88855f9f7784502a7f,3369291853724898216,3.369291853724898216
6,25373918,0x4e07628417fa726763ba9bfd8fd9bf719c183a7f611e...,35,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0xba47cbfdd61029833841fcaa2ec2591ddfa87e51,0x51c72848c68a965f66fa7a88855f9f7784502a7f,2471445342369000000,2.471445342369
7,25373918,0x47573d591726ab11d8e9f20299485eea411bb80fbdda...,39,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0xe0554a476a092703abdb3ef35c80e0d76d32939f,0x51c72848c68a965f66fa7a88855f9f7784502a7f,1003611932675678005,1.003611932675678005
8,25373918,0x86d7b2b5980e3e261478f7b683bfd680a22737ab19b4...,69,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0xeb98191df80179a36ca7026ee10c41d2606d90f5,0xb2896002662372b95086a4fcaaf7dfa6c7727b4a,193876253734702886,0.193876253734702886
9,25373918,0x095308efcbefabc14ffff714c90b990f3de6b75dc3b4...,116,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x7f54f05635d15cde17a49502fedb9d1803a3be8a,0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640,4854258920000000000,4.85425892
